In [ ]:
from datasets import load_dataset               # datasets is the huggingface Datasets library

In [3]:
#The WikiText language modeling dataset is a collection of over 100 million tokens 
# extracted from the set of verified Good and Featured articles on Wikipedia.

### Loading Dataset

In [14]:
dataset = load_dataset(
    "Salesforce/wikitext",             # dataset_name
    "wikitext-2-raw-v1"                # dataset configuration
)

In [4]:
dataset

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [10]:
dataset["train"][0]

{'text': ''}

In [11]:
dataset["train"][1]

{'text': ' = Valkyria Chronicles III = \n'}

In [ ]:
for i in range(10):
    print(f"ROW {i} : {repr(dataset['train'][i]['text'])}")            # takes the i'th row and repr() shows the raw representation, including things like \n and empty strings

ROW 0 : ''
ROW 1 : ' = Valkyria Chronicles III = \n'
ROW 2 : ''
ROW 3 : ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
ROW 4 : " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments ,

### <span style='color:green'> **Word Level Tokenizer (Manual)**

In [ ]:
text = dataset['train'][1]['text']
print(text)

tokens = text.split()                         # splits a string into words using a whitespace by default
print(tokens)                                 # this is called a tokenizer

 = Valkyria Chronicles III = 

['=', 'Valkyria', 'Chronicles', 'III', '=']


In [ ]:
example = "Hello, world! The patient's knee hurts."

print(example.split())               # our naive tokenizer attached the punctuation to words i.e. "Hello" and "Hello," would be considered different words
                                     # essentially a very basic whitespace tokenizer

['Hello,', 'world!', 'The', "patient's", 'knee', 'hurts.']


In [ ]:
## segregate punctuation

import re                            # pythons regular expression library

example = "Hello, world! The patient's knee hurts."
tokens = re.findall(r"\w+|[^\w\s]", example)               # \w+ means on or more word characters eg Hello
                                                           # | means or
                                                           # [] defines a character class
                                                           # ^ means NOT when immediately appears after [
                                                           # \w means a word character
                                                           # \s means a whitespace character
                                                           # [^\w\s] means match one character that is not a word character and not a whitespace
print(tokens)                        # punctuation becomes its own character

['Hello', ',', 'world', '!', 'The', 'patient', "'", 's', 'knee', 'hurts', '.']


In [ ]:
example = "The patient saw the doctor. THE doctor responded."

tokens = re.findall(r"\w+|[^\w\s]", example.lower())        # we lose away capitalization info : US > us
                                                            # therefore tokenizers can be described as "cased" or "uncased"
print(tokens)

['the', 'patient', 'saw', 'the', 'doctor', '.', 'the', 'doctor', 'responded', '.']


In [20]:
## building the tokenization function

import re

def word_tokenize(text):
    text = text.lower()
    return re.findall(r"\w+|[^\w\s]", text)


In [21]:
word_tokenize("The patients knees hurt")

['the', 'patients', 'knees', 'hurt']

In [ ]:
## count every token in WikiText-2
from collections import Counter                     # pythons own counter

token_counts = Counter()

for row in dataset['train']:
    tokens = word_tokenize(row['text'])             # looks at each token, if exists update the count with +=1
    token_counts.update(tokens)

In [23]:
print("Total unique tokens:", len(token_counts))
print("Total tokens:", sum(token_counts.values()))

Total unique tokens: 65989
Total tokens: 2122137


In [24]:
token_counts.most_common(20)

[('the', 130771),
 (',', 102624),
 ('.', 84291),
 ('of', 57032),
 ('and', 50738),
 ('@', 45600),
 ('in', 45019),
 ('to', 39522),
 ('a', 36567),
 ('=', 29570),
 ('"', 28309),
 ('was', 21008),
 ("'", 18655),
 ('-', 17337),
 ('on', 15141),
 ('as', 15058),
 ('s', 14982),
 ('that', 14351),
 ('for', 13795),
 ('with', 13012)]

In [25]:
rare_tokens = [
    token
    for token, count in token_counts.items()
    if count == 1
]

print("Tokens appearing exactly once:", len(rare_tokens))
print(rare_tokens[:30])

Tokens appearing exactly once: 26843
['calamaty', 'forgiving', 'unvoiced', 'scanned', 'boosts', 'depleting', 'reila', 'shocktroopers', 'thereon', 'altaha', 'abilia', 'marcellis', 'jinxed', 'deniability', 'gusurg', 'hounded', 'randgriz', 'yamanobe', 'koichi', 'kishiko', 'miyagi', 'seiki', 'nagakawa', 'takayuki', 'shouji', 'redid', 'redoing', 'atonality', 'mitsuhiro', 'ohta']


In [27]:
# Approx 40% of our tokens appear exactly once
# Total vocab size = 66k x 768 (assume rep vector to be dimension 768) = 50.68M
# at FP32 50.68M x 4 bytes = 203MB just for the token embedding matrix

In [26]:
## lets create actual vocabulary

special_tokens = ['[PAD]', '[UNK]']

vocab = {
    token: idx 
    for idx, token in enumerate(special_tokens)           # enumerate gives each token an index
}

for token, count in token_counts.most_common():           # most_common() returns the tokens ordered from most to least frequent
    vocab[token] = len(vocab)                             # each token gets the next available ID

In [29]:
print("Vocabulary size:", len(vocab))
list(vocab.items())[:20]

Vocabulary size: 65991


[('[PAD]', 0),
 ('[UNK]', 1),
 ('the', 2),
 (',', 3),
 ('.', 4),
 ('of', 5),
 ('and', 6),
 ('@', 7),
 ('in', 8),
 ('to', 9),
 ('a', 10),
 ('=', 11),
 ('"', 12),
 ('was', 13),
 ("'", 14),
 ('-', 15),
 ('on', 16),
 ('as', 17),
 ('s', 18),
 ('that', 19)]

### Encode

In [30]:
## Now we will encode the actual text to token id's we have assigned above

def encode(text):
    tokens = word_tokenize(text)                    # takes a string and tokens it using our custome built tokenizer

    token_ids = [
        vocab.get(token, vocab['[UNK]'])            # looks up each token in vocab, if exists returns its id, else return the id of unk
        for token in tokens
    ]

    return tokens, token_ids

In [32]:
tokens, ids = encode(
    "The patient has osteochondritis and chondromalacia."
)

print("Tokens:", tokens)
print("IDs:", ids)

Tokens: ['the', 'patient', 'has', 'osteochondritis', 'and', 'chondromalacia', '.']
IDs: [2, 8658, 54, 1, 6, 1, 4]


In [33]:
encode("The patient has patellofemoral chondromalacia.")

(['the', 'patient', 'has', 'patellofemoral', 'chondromalacia', '.'],
 [2, 8658, 54, 1, 1, 4])

In [ ]:
## We see that the problem with above is medical terms like 'patellofemoral' don't exist
## we replace them with unk and info about that is lost, all receive the exact same embedding

### <span style='color:green'> **Character level tokenizer** (manual)

In [ ]:
from collections import Counter

char_counts = Counter()

for row in dataset['train']:                         # because text is a string, Python treats it as an iterable of individual characters
    text = row['text'].lower()
    char_counts.update(text)

In [36]:
print("Unique characters:", len(char_counts))
print("Total characters:", sum(char_counts.values()))

char_counts.most_common(30)

Unique characters: 940
Total characters: 10892991


[(' ', 2075726),
 ('e', 1001029),
 ('t', 725515),
 ('a', 716871),
 ('i', 609791),
 ('n', 605384),
 ('o', 595295),
 ('r', 547660),
 ('s', 544866),
 ('h', 404479),
 ('l', 340389),
 ('d', 339004),
 ('c', 271811),
 ('u', 215523),
 ('m', 214359),
 ('f', 184661),
 ('g', 168439),
 ('p', 166824),
 ('w', 142745),
 ('b', 130070),
 ('y', 126667),
 (',', 102624),
 ('v', 86098),
 ('.', 84291),
 ('k', 56452),
 ('@', 45600),
 ('1', 36161),
 ('0', 32680),
 ('=', 29570),
 ('"', 28309)]

In [37]:
## build vocabulary as before
special_tokens = ['[PAD]', '[UNK]']

char_vocab = {
    char: idx 
    for idx, char in enumerate(special_tokens)
}

for char, count in char_counts.most_common():
    char_vocab[char] = len(char_vocab)

print("Character vocabulary size:", len(char_vocab))

Character vocabulary size: 942


### Encode

In [39]:
def char_encode(text):
    text = text.lower()

    characters = list(text)

    ids = [
        char_vocab.get(char, char_vocab['[UNK]'])
        for char in characters
    ]

    return characters, ids

In [40]:
sentence = "The patient has patellofemoral chondromalacia."

characters, char_ids = char_encode(sentence)

print("Characters:")
print(characters)

print("\nIDs:")
print(char_ids)

print("\nNumber of tokens:", len(characters))

Characters:
['t', 'h', 'e', ' ', 'p', 'a', 't', 'i', 'e', 'n', 't', ' ', 'h', 'a', 's', ' ', 'p', 'a', 't', 'e', 'l', 'l', 'o', 'f', 'e', 'm', 'o', 'r', 'a', 'l', ' ', 'c', 'h', 'o', 'n', 'd', 'r', 'o', 'm', 'a', 'l', 'a', 'c', 'i', 'a', '.']

IDs:
[4, 11, 3, 2, 19, 5, 4, 6, 3, 7, 4, 2, 11, 5, 10, 2, 19, 5, 4, 3, 12, 12, 8, 17, 3, 16, 8, 9, 5, 12, 2, 14, 11, 8, 7, 13, 9, 8, 16, 5, 12, 5, 14, 6, 5, 25]

Number of tokens: 46


In [41]:
word_tokens, word_ids = encode(sentence)

print("WORD TOKENIZER")
print("Tokens:", word_tokens)
print("Number of tokens:", len(word_tokens))

print("\nCHARACTER TOKENIZER")
print("Tokens:", characters)
print("Number of tokens:", len(characters))

WORD TOKENIZER
Tokens: ['the', 'patient', 'has', 'patellofemoral', 'chondromalacia', '.']
Number of tokens: 6

CHARACTER TOKENIZER
Tokens: ['t', 'h', 'e', ' ', 'p', 'a', 't', 'i', 'e', 'n', 't', ' ', 'h', 'a', 's', ' ', 'p', 'a', 't', 'e', 'l', 'l', 'o', 'f', 'e', 'm', 'o', 'r', 'a', 'l', ' ', 'c', 'h', 'o', 'n', 'd', 'r', 'o', 'm', 'a', 'l', 'a', 'c', 'i', 'a', '.']
Number of tokens: 46


### <span style='color:green'> **BPE Tokenizer** (manual)

In [60]:
small_corpus = [
    "low",
    "low",
    "low",
    "low",
    "low",
    "lower",
    "lower",
    "newest",
    "newest",
    "newest",
    "newest",
    "newest",
    "newest",
    "widest",
    "widest",
    "widest"
]

In [61]:
word_splits = {}

for word in set(small_corpus):                  # set removes duplicate words
    word_splits[word] = list(word)              # list word splits the string into characters

word_splits

{'newest': ['n', 'e', 'w', 'e', 's', 't'],
 'widest': ['w', 'i', 'd', 'e', 's', 't'],
 'lower': ['l', 'o', 'w', 'e', 'r'],
 'low': ['l', 'o', 'w']}

In [62]:
## BPE needs to know the frequency
from collections import Counter

word_freqs = Counter(small_corpus)

print(word_freqs)

Counter({'newest': 6, 'low': 5, 'widest': 3, 'lower': 2})


In [63]:
## Count adjacent pairs
def compute_pair_freqs(word_splits, word_freqs):
        pair_freqs = Counter()                               # creates an empty counter for pairs

        for word, split in word_splits.items():              # pull the word and it's frequency
            frequency = word_freqs[word]         

            for i in range(len(split)-1):                    # then for each adjacent pair of characters in the word, add frequency using the counter, since that is the number of times those pairs occur together
                  pair = (split[i], split[i+1])

                  pair_freqs[pair] += frequency

        return pair_freqs

In [64]:
pair_freqs = compute_pair_freqs(word_splits, word_freqs)
pair_freqs

Counter({('e', 's'): 9,
         ('s', 't'): 9,
         ('w', 'e'): 8,
         ('l', 'o'): 7,
         ('o', 'w'): 7,
         ('n', 'e'): 6,
         ('e', 'w'): 6,
         ('w', 'i'): 3,
         ('i', 'd'): 3,
         ('d', 'e'): 3,
         ('e', 'r'): 2})

In [65]:
best_pair = max(pair_freqs, key=pair_freqs.get)                # normally max(pair_freqs) would compare the keys
                                           # but key=pair_freqs.get tells max() compare each pair using its value/frequency

print("Best pair:", best_pair)
print("Frequency:", pair_freqs[best_pair])

Best pair: ('e', 's')
Frequency: 9


In [66]:
## BPE Merge
def merge_pair(a,b,word_splits):
    new_splits = {}

    for word, split in word_splits.items():                # go through each word and its current tokens
        new_split = []                                     # this will store the token
        i = 0                                              # while i tracks our current position

        while i < len(split):                              # while loop examines tokens one by one
            if (
                i < len(split) - 1                         # there should be another token after the current one
                and split[i] == a
                and split[i+1] == b
            ):
                new_split.append(a+b)
                i += 2                                     # we then skip both tokens because they have already been merged

            else:
                new_split.append(split[i])
                i += 1

        new_splits[word] = new_split

    return new_splits

In [67]:
word_splits = merge_pair("e", "s", word_splits)

word_splits

{'newest': ['n', 'e', 'w', 'es', 't'],
 'widest': ['w', 'i', 'd', 'es', 't'],
 'lower': ['l', 'o', 'w', 'e', 'r'],
 'low': ['l', 'o', 'w']}

In [68]:
pair_freqs = compute_pair_freqs(word_splits, word_freqs)

print(pair_freqs)

Counter({('es', 't'): 9, ('l', 'o'): 7, ('o', 'w'): 7, ('n', 'e'): 6, ('e', 'w'): 6, ('w', 'es'): 6, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'es'): 3, ('w', 'e'): 2, ('e', 'r'): 2})


In [70]:
## after one iteration we managed to merge one together, we need to run this in a loop

In [69]:
pair_freqs = compute_pair_freqs(word_splits, word_freqs)

print(pair_freqs)

best_pair = max(pair_freqs, key=pair_freqs.get)

print("\nBest pair:", best_pair)
print("Frequency:", pair_freqs[best_pair])

Counter({('es', 't'): 9, ('l', 'o'): 7, ('o', 'w'): 7, ('n', 'e'): 6, ('e', 'w'): 6, ('w', 'es'): 6, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'es'): 3, ('w', 'e'): 2, ('e', 'r'): 2})

Best pair: ('es', 't')
Frequency: 9


### <span style='color:green'> **WordPiece** (Manual)

In [71]:
word_freqs

Counter({'newest': 6, 'low': 5, 'widest': 3, 'lower': 2})

In [ ]:
## reset our word piece
word_splits_wp = {
    word: list(word)
    for word in word_freqs
}

word_splits_wp

{'low': ['l', 'o', 'w'],
 'lower': ['l', 'o', 'w', 'e', 'r'],
 'newest': ['n', 'e', 'w', 'e', 's', 't'],
 'widest': ['w', 'i', 'd', 'e', 's', 't']}

In [ ]:
## Diff - BPE only counted pairs while WP will also look at word pair association strength

In [73]:
## calculate token and pair frequencies
def compute_wordpiece_stats(word_splits, word_freqs):

    token_freqs = Counter()
    pair_freqs = Counter()

    for word, split in word_splits.items():                  # basically take the word and it's characters

        frequency = word_freqs[word]                         # lookup the frequency of that word from the word_freqs dictionary & store it in a var

        # count individual tokens
        for token in split:                                  # now loop through the tokens 1 by 1
            token_freqs[token] += frequency                  # in token_freq add that token and update the frequency by the count of the original word that token appears in 

        # count adjacent pairs
        for i in range(len(split) - 1):                      # also look at that token and the next one as a pair
            pair = (split[i], split[i+1])      
            pair_freqs[pair] += frequency                    # add a count in the pair_freq using the count of the original word that pair is from 

    return token_freqs, pair_freqs

In [74]:
token_freqs, pair_freqs = compute_wordpiece_stats(
    word_splits_wp,
    word_freqs
)

In [75]:
print("TOKEN FREQUENCIES")
print(token_freqs)

print("\nPAIR FREQUENCIES")
print(pair_freqs)

TOKEN FREQUENCIES
Counter({'e': 17, 'w': 16, 's': 9, 't': 9, 'l': 7, 'o': 7, 'n': 6, 'i': 3, 'd': 3, 'r': 2})

PAIR FREQUENCIES
Counter({('e', 's'): 9, ('s', 't'): 9, ('w', 'e'): 8, ('l', 'o'): 7, ('o', 'w'): 7, ('n', 'e'): 6, ('e', 'w'): 6, ('w', 'i'): 3, ('i', 'd'): 3, ('d', 'e'): 3, ('e', 'r'): 2})


In [79]:
## calculating scores for each pair
pair_scores = {}

for pair, pair_count in pair_freqs.items():             # pull out the pairs

    a,b = pair                                          # unnest the list

    score = (                                           # calculate score using pair and individual count
        pair_count
        /
        (token_freqs[a] * token_freqs[b])
    )

    pair_scores[pair] = score                           # add to the dict

In [80]:
sorted_scores = sorted(
    pair_scores.items(),                                # converts the dictionary into (pair, score) pairs
    key=lambda x: x[1],                                 # for each item x use its second element i.e. score above
    reverse=True                                        # and sort largest > smallest
)

for pair, score in sorted_scores:                      
    print(pair, round(score, 4))

('i', 'd') 0.3333
('l', 'o') 0.1429
('s', 't') 0.1111
('o', 'w') 0.0625
('w', 'i') 0.0625
('e', 'r') 0.0588
('n', 'e') 0.0588
('e', 's') 0.0588
('d', 'e') 0.0588
('w', 'e') 0.0294
('e', 'w') 0.0221


In [81]:
## BPE Merge
def merge_pair(a,b,word_splits):
    new_splits = {}

    for word, split in word_splits.items():                # go through each word and its current tokens
        new_split = []                                     # this will store the token
        i = 0                                              # while i tracks our current position

        while i < len(split):                              # while loop examines tokens one by one
            if (
                i < len(split) - 1                         # there should be another token after the current one
                and split[i] == a
                and split[i+1] == b
            ):
                new_split.append(a+b)
                i += 2                                     # we then skip both tokens because they have already been merged

            else:
                new_split.append(split[i])
                i += 1

        new_splits[word] = new_split

    return new_splits

In [82]:
word_splits = merge_pair("i", "d", word_splits)

word_splits

{'newest': ['n', 'e', 'w', 'es', 't'],
 'widest': ['w', 'id', 'es', 't'],
 'lower': ['l', 'o', 'w', 'e', 'r'],
 'low': ['l', 'o', 'w']}